# Raw Ingestion — Food Inspections (dev)

Lands `raw/<ingest-date>/food_inspections.csv` from the `foodguard_ai.raw.food_inspections` volume into the **bronze Delta table** `foodguard_ai.bronze.food_inspections`, **as-is**: every column is read as STRING (no type conversion, no cleaning, no filtering).

Audit columns appended at write time: `_source` (source file name), `_ingested_at` (UTC time the file was received), `_ingest_date` (date folder). The table is **append-only** — every ingest run appends its batch; exact duplicates across runs are expected and are resolved downstream (silver dedups on `record_id`). Raw history stays queryable forever.

In [0]:
# Source file (raw volume) and bronze Delta table.
from datetime import date, datetime, timezone

INGEST_DATE = date.today().isoformat()
SOURCE_CSV = f"/Volumes/foodguard_ai/raw/food_inspections/{INGEST_DATE}/food_inspections.csv"
BRONZE_TABLE = "foodguard_ai.bronze.food_inspections"

# Audit metadata: which file this came from, and when it was received (UTC).
SOURCE_FILE = SOURCE_CSV.rsplit("/", 1)[-1]
INGESTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Source : {SOURCE_CSV}")
print(f"Target : {BRONZE_TABLE}")
print(f"Audit  : source={SOURCE_FILE} | ingested_at={INGESTED_AT} | ingest_date={INGEST_DATE}")

In [0]:
# Raw read: no schema, no inference — every column stays exactly as in the CSV (STRING).
# multiLine=True keeps records intact: some quoted fields (violations, location) contain embedded newlines.
raw_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(SOURCE_CSV)
)

raw_df.printSchema()
print(f"Rows read: {raw_df.count()}")
display(raw_df.limit(5))

In [0]:
# Append to the bronze Delta table: original columns as-is (all STRING), plus audit columns.
# Append-only raw landing — re-running with the same file re-appends; silver dedups by record_id.
from pyspark.sql import functions as F

bronze_df = (
    raw_df
    .withColumn("_source", F.lit(SOURCE_FILE))
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_ingest_date", F.lit(INGEST_DATE))
)

(
    bronze_df.write
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Appended {bronze_df.count()} rows to {BRONZE_TABLE} (ingest_date={INGEST_DATE})")

In [0]:
# Sanity check: confirm today's batch landed in the table with audit columns intact.
from pyspark.sql import functions as F

table_df = spark.table(BRONZE_TABLE)
today_df = table_df.where(F.col("_ingest_date") == INGEST_DATE)
# A single ingest date can hold multiple appended batches (re-running the same
# CSV re-appends by design), so verify THIS run's batch via its _ingested_at.
run_df = table_df.where(F.col("_ingested_at") == INGESTED_AT)

src_count = raw_df.count()
today_count = run_df.count()
same_schema = raw_df.schema == today_df.drop("_source", "_ingested_at", "_ingest_date").schema
src_values = [r["_source"] for r in today_df.select("_source").distinct().collect()]
null_audits = today_df.where(
    F.col("_ingested_at").isNull() | F.col("_ingest_date").isNull()
).count()
audit_ok = src_values == [SOURCE_FILE] and null_audits == 0

print(f"Rows  — source CSV: {src_count} | bronze (this run): {today_count} | match: {src_count == today_count}")
print(f"Schema match (all STRING, minus audit cols): {same_schema}")
print(f"Audit — _source: {src_values} | null audit rows: {null_audits}")
assert src_count == today_count, "Row count mismatch!"
assert same_schema, "Schema mismatch — types were altered!"
assert audit_ok, "Audit columns missing or incorrect!"
print(f"Bronze table total (all ingest dates): {table_df.count()}")
display(today_df.limit(5))